# 2035 Full-Year PCM Analysis

**Scenario:** GTEP Stage 3 solution — 278 generators, 123 buses, PTDF formulation
**Period:** January 1 – December 31, 2035 (365 days, 8760 hours)
**Simulation:** Prescient UC+ED with day-ahead RUC and real-time SCED

---

> **Note:** This analysis uses `gen.csv` with the corrected `HR_avg_0` values.
> The previous run showed zero NUC/COAL dispatch due to astronomically large
> `HR_avg_0` values inflating startup costs. The fix was validated in a 30-day
> sanity check before this full-year run.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 5)})

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_ROOT = Path(os.path.abspath('')).parent / 'data' / 'retirement_allowed_no_extreme_half_load_local'
RESULTS_DIR = DATA_ROOT / 'Prescient_2_2035' / 'results'
GEN_CSV = DATA_ROOT / 'Prescient_2_2035' / 'gen.csv'
OUTPUT_DIR = Path(os.path.abspath(''))
OUTPUT_SUFFIX = '_2035'

print(f'Results dir: {RESULTS_DIR}')
print(f'Results exist: {RESULTS_DIR.exists()}')
print(f'Output dir: {OUTPUT_DIR}')


def _prescient_output_to_df(file_name):
    """Load Prescient output CSV and combine Date/Hour/Minute into Datetime."""
    df = pd.read_csv(file_name)
    if 'Minute' in df.columns:
        df['Datetime'] = (
            pd.to_datetime(df['Date'])
            + pd.to_timedelta(df['Hour'], 'hour')
            + pd.to_timedelta(df['Minute'], 'minute')
        )
        df.drop(columns=['Date', 'Hour', 'Minute'], inplace=True)
    elif 'Hour' in df.columns:
        df['Datetime'] = (
            pd.to_datetime(df['Date'])
            + pd.to_timedelta(df['Hour'], 'hour')
        )
        df.drop(columns=['Date', 'Hour'], inplace=True)
    else:
        df['Datetime'] = pd.to_datetime(df['Date'])
        df.drop(columns=['Date'], inplace=True)
    cols = df.columns.tolist()
    cols = cols[-1:] + cols[:-1]
    return df[cols]


def make_lmp_csv(lmp_path, bus_details_path, bus_name, output_dir='.'):
    """Extract LMP for a single bus and append to the wide-format LMP CSV."""
    out_csv = os.path.join(output_dir, f'Bus_LMP{OUTPUT_SUFFIX}.csv')
    bdf = _prescient_output_to_df(bus_details_path)
    bdf = bdf[bdf['Bus'] == bus_name][['Datetime', 'LMP', 'LMP DA']]
    bdf.set_index('Datetime', inplace=True)
    bdf = bdf.rename(columns={'LMP': f'{bus_name}_LMP', 'LMP DA': f'{bus_name}_LMP DA'})

    if lmp_path is None:
        bdf.to_csv(out_csv)
    else:
        lmp_df = pd.read_csv(lmp_path).set_index('Datetime')
        if f'{bus_name}_LMP' in lmp_df.columns:
            return
        bdf_aligned = bdf.reindex(lmp_df.index)
        lmp_df = pd.concat([lmp_df, bdf_aligned], axis=1)
        lmp_df.to_csv(out_csv)


def make_dispatch_csv(dispatch_path, gen_details_path, gen_name, gen_type,
                      other_info=None, output_dir='.'):
    """Extract dispatch for a single generator and append to wide-format CSV."""
    out_csv = os.path.join(output_dir, f'Generator_Dispatch{OUTPUT_SUFFIX}.csv')
    gdf = _prescient_output_to_df(gen_details_path)

    if gen_type == 'fossil':
        info_list = ['Datetime', 'Dispatch', 'Dispatch DA']
    elif gen_type == 'renew':
        info_list = ['Datetime', 'Output', 'Output DA']
    else:
        raise ValueError(f'Unknown gen_type: {gen_type}')

    if other_info is not None:
        info_list.extend(other_info)

    gdf_filtered = gdf[gdf['Generator'] == gen_name]
    if len(gdf_filtered) == 0:
        try:
            gdf_filtered = gdf[gdf['Generator'] == int(gen_name)]
        except (ValueError, TypeError):
            pass
    if len(gdf_filtered) == 0:
        print(f'WARNING: Generator {gen_name} not found')
        return

    gdf_filtered = gdf_filtered[info_list]
    gdf_filtered.set_index('Datetime', inplace=True)
    new_col_name = {col: f'{gen_name}_{col}' for col in info_list if col != 'Datetime'}
    gdf_filtered = gdf_filtered.rename(columns=new_col_name)

    if dispatch_path is None:
        gdf_filtered.to_csv(out_csv)
    else:
        dispatch_df = pd.read_csv(dispatch_path).set_index('Datetime')
        first_col = list(new_col_name.values())[0]
        if first_col in dispatch_df.columns:
            return
        gdf_aligned = gdf_filtered.reindex(dispatch_df.index)
        dispatch_df = pd.concat([dispatch_df, gdf_aligned], axis=1)
        dispatch_df.to_csv(out_csv)

## 1. Load Raw Data

In [ ]:
# ── Load simulation outputs ───────────────────────────────────────────────
overall = pd.read_csv(RESULTS_DIR / 'overall_simulation_output.csv')
daily = pd.read_csv(RESULTS_DIR / 'daily_summary.csv', parse_dates=['Date'])
hourly = _prescient_output_to_df(RESULTS_DIR / 'hourly_summary.csv')
bus_df = _prescient_output_to_df(RESULTS_DIR / 'bus_detail.csv')
therm = _prescient_output_to_df(RESULTS_DIR / 'thermal_detail.csv')
renew = _prescient_output_to_df(RESULTS_DIR / 'renewables_detail.csv')
gen = pd.read_csv(GEN_CSV)

# ── Dynamic dimensions ────────────────────────────────────────────────────
N_DAYS = 365
N_HOURS = N_DAYS * 24  # 8760
N_BUSES = 123
N_THERMAL = len(gen[~gen['Unit Type'].isin(['WIND', 'PV', 'HYDRO'])])
N_RENEW = len(gen[gen['Unit Type'].isin(['WIND', 'PV', 'HYDRO'])])

print(f'Overall:       {len(overall):>12,} rows')
print(f'Daily:         {len(daily):>12,} rows  (expected {N_DAYS})')
print(f'Hourly:        {len(hourly):>12,} rows  (expected {N_HOURS:,})')
print(f'Bus detail:    {len(bus_df):>12,} rows  (expected {N_BUSES * N_HOURS:,})')
print(f'Thermal:       {len(therm):>12,} rows  (expected {N_THERMAL * N_HOURS:,})')
print(f'Renewables:    {len(renew):>12,} rows  (expected {N_RENEW * N_HOURS:,})')
print(f'Generators:    {len(gen):>12,} rows  (expected 278)')

# Validate dimensions
assert len(daily) == N_DAYS, f'Expected {N_DAYS} daily rows, got {len(daily)}'
assert len(hourly) == N_HOURS, f'Expected {N_HOURS} hourly rows, got {len(hourly)}'
assert len(bus_df) == N_BUSES * N_HOURS, f'Expected {N_BUSES * N_HOURS:,} bus rows, got {len(bus_df)}'
assert len(therm) == N_THERMAL * N_HOURS, f'Expected {N_THERMAL * N_HOURS:,} thermal rows, got {len(therm)}'
assert len(renew) == N_RENEW * N_HOURS, f'Expected {N_RENEW * N_HOURS:,} renew rows, got {len(renew)}'
assert len(gen) == 278, f'Expected 278 generators, got {len(gen)}'
print('\nAll dimension checks passed.')

## 2. HR_avg_0 Fix Verification

The `HR_avg_0` column was corrected in `gen.csv`. Previously, NUC and COAL had
astronomically large values (NUC: 3.6M–4.0M, COAL: 35K–309K) making them uneconomic.
After the fix, these generators should now dispatch normally.

In [ ]:
# ── Verify HR_avg_0 fix — NUC/COAL should now dispatch ────────────────────
hr_check = gen[['GEN UID', 'Unit Type', 'Fuel', 'Fuel Price $/MMBTU',
                'PMin MW', 'PMax MW', 'HR_avg_0', 'HR_incr_1']].copy()
hr_check['Cost_at_PMin'] = (
    hr_check['Fuel Price $/MMBTU'] * hr_check['HR_avg_0'] * 0.001 * hr_check['PMin MW']
)
hr_check['Expected_MC'] = hr_check['Fuel Price $/MMBTU'] * hr_check['HR_incr_1'] * 0.001

print('HR_avg_0 by Unit Type (sample — should now be reasonable):')
print('=' * 90)
for utype in ['NUC', 'COAL', 'CT']:
    subset = hr_check[hr_check['Unit Type'] == utype].head(3)
    if len(subset) > 0:
        print(f'\n{utype}:')
        print(subset[['GEN UID', 'HR_avg_0', 'HR_incr_1', 'Cost_at_PMin',
                       'Expected_MC']].to_string(index=False))

# ── Check NUC and COAL dispatch ───────────────────────────────────────────
nuc_ids = gen[gen['Unit Type'] == 'NUC']['GEN UID'].values
coal_ids = gen[gen['Unit Type'] == 'COAL']['GEN UID'].values
nuc_dispatch = therm[therm['Generator'].isin(nuc_ids)]['Dispatch'].sum()
coal_dispatch = therm[therm['Generator'].isin(coal_ids)]['Dispatch'].sum()

print(f'\nNUC total dispatch over {N_DAYS} days: {nuc_dispatch:,.1f} MWh')
print(f'COAL total dispatch over {N_DAYS} days: {coal_dispatch:,.1f} MWh')
print(f'NUC generators: {len(nuc_ids)}, COAL generators: {len(coal_ids)}')

if nuc_dispatch > 0:
    nuc_cf = nuc_dispatch / (gen[gen['Unit Type'] == 'NUC']['PMax MW'].sum() * N_HOURS) * 100
    print(f'\nNUC capacity factor: {nuc_cf:.1f}% — {"baseload as expected" if nuc_cf > 50 else "low — investigate"}')
else:
    print('\nWARNING: NUC dispatch is still zero — HR_avg_0 fix may not have taken effect')

if coal_dispatch > 0:
    coal_cf = coal_dispatch / (gen[gen['Unit Type'] == 'COAL']['PMax MW'].sum() * N_HOURS) * 100
    print(f'COAL capacity factor: {coal_cf:.1f}%')
else:
    print('NOTE: COAL dispatch is zero — may be economically displaced by NUC + CT')

## 3. Annual Metrics Dashboard

In [ ]:
# ── Parse overall metrics ─────────────────────────────────────────────────
o = overall.iloc[0]

# Load-weighted LMP
_LOW_DEMAND_MW = 1.0
bus_df['_wt_lmp'] = bus_df['Demand'] * bus_df['LMP DA']
_total_demand = bus_df['Demand'].sum()
if _total_demand < _LOW_DEMAND_MW:
    warnings.warn('Total demand < 1 MW — using simple mean as fallback')
    lw_lmp = bus_df['LMP DA'].mean()
else:
    lw_lmp = bus_df['_wt_lmp'].sum() / _total_demand
bus_df.drop(columns='_wt_lmp', inplace=True)

# Dashboard
metrics = {
    'Total Demand (TWh)': o['Total demand'] / 1e6,
    'Total Fixed Costs ($B)': o['Total fixed costs'] / 1e9,
    'Total Generation Costs ($B)': o['Total generation costs'] / 1e9,
    'Total Costs ($B)': o['Total costs'] / 1e9,
    'Load-Weighted LMP ($/MWh)': lw_lmp,
    'Cumulative Avg Price ($/MWh)': o['Cumulative average price'],
    'Renewables Penetration (%)': o['Overall renewables penetration rate'],
    'Total Load Shedding (MWh)': o['Total load shedding'],
    'Total Curtailment (MWh)': o['Total renewables curtailment'],
    'Total Reserve Shortfall (MWh)': o['Total reserve shortfall'],
    'Peak Demand (MW)': o['Maximum observed demand'],
    'Total On/Offs': o['Total on/offs'],
    'Total Energy Payments ($B)': o['Total energy payments'] / 1e9,
    'Total Reserve Payments ($M)': o['Total reserve payments'] / 1e6,
}

print(f'2035 {N_DAYS}-Day Metrics Dashboard')
print('=' * 55)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k:<40s} {v:>12,.2f}')
    else:
        print(f'  {k:<40s} {v:>12,}')

# Sanity flags
print('\n── Sanity Flags ──')
flags = []
if o['Total load shedding'] > 0:
    flags.append(f'LOAD SHEDDING: {o["Total load shedding"]:.2f} MWh')
if o['Total renewables curtailment'] > 0:
    flags.append(f'CURTAILMENT: {o["Total renewables curtailment"]:.2f} MWh')
if lw_lmp < 10 or lw_lmp > 60:
    flags.append(f'LMP out of normal range: ${lw_lmp:.2f}/MWh')
if o['Total reserve shortfall'] > 100:
    flags.append(f'RESERVE SHORTFALL: {o["Total reserve shortfall"]:.1f} MWh')
if not flags:
    print('  All basic checks pass')
else:
    for f in flags:
        print(f'  {f}')

## 4. Extract Standardized Output Files

In [ ]:
# ── Build Bus_LMP_2035.csv ────────────────────────────────────────────────
bus_names = sorted(bus_df['Bus'].unique())
print(f'Building Bus_LMP{OUTPUT_SUFFIX}.csv for {len(bus_names)} buses...')

lmp_csv = str(OUTPUT_DIR / f'Bus_LMP{OUTPUT_SUFFIX}.csv')
bus_detail_path = str(RESULTS_DIR / 'bus_detail.csv')

for idx, bus_name in enumerate(bus_names):
    if idx == 0:
        make_lmp_csv(lmp_path=None, bus_details_path=bus_detail_path,
                     bus_name=bus_name, output_dir=str(OUTPUT_DIR))
    else:
        make_lmp_csv(lmp_path=lmp_csv, bus_details_path=bus_detail_path,
                     bus_name=bus_name, output_dir=str(OUTPUT_DIR))
    if (idx + 1) % 30 == 0:
        print(f'  ... processed {idx + 1}/{len(bus_names)} buses')

df_lmp_check = pd.read_csv(lmp_csv)
print(f'\nBus_LMP{OUTPUT_SUFFIX}.csv: {df_lmp_check.shape[0]} rows x {df_lmp_check.shape[1]} cols')
print(f'Expected: {N_HOURS} rows x {len(bus_names) * 2 + 1} cols')

# ── Build Generator_Dispatch_2035.csv ─────────────────────────────────────
thermal_gens = sorted(therm['Generator'].unique())
renew_gens = sorted(renew['Generator'].unique())
print(f'\nBuilding Generator_Dispatch{OUTPUT_SUFFIX}.csv for {len(thermal_gens)} thermal + {len(renew_gens)} renewable gens...')

dispatch_csv = str(OUTPUT_DIR / f'Generator_Dispatch{OUTPUT_SUFFIX}.csv')
thermal_detail_path = str(RESULTS_DIR / 'thermal_detail.csv')
renew_detail_path = str(RESULTS_DIR / 'renewables_detail.csv')

thermal_other_info = ['Unit Cost', 'Unit State']
for idx, gn in enumerate(thermal_gens):
    if idx == 0:
        make_dispatch_csv(dispatch_path=None, gen_details_path=thermal_detail_path,
                          gen_name=gn, gen_type='fossil',
                          other_info=thermal_other_info, output_dir=str(OUTPUT_DIR))
    else:
        make_dispatch_csv(dispatch_path=dispatch_csv, gen_details_path=thermal_detail_path,
                          gen_name=gn, gen_type='fossil',
                          other_info=thermal_other_info, output_dir=str(OUTPUT_DIR))
    if (idx + 1) % 30 == 0:
        print(f'  ... processed {idx + 1}/{len(thermal_gens)} thermal gens')

renew_other_info = ['Curtailment']
for gn in renew_gens:
    make_dispatch_csv(dispatch_path=dispatch_csv, gen_details_path=renew_detail_path,
                      gen_name=gn, gen_type='renew',
                      other_info=renew_other_info, output_dir=str(OUTPUT_DIR))

df_disp_check = pd.read_csv(dispatch_csv)
print(f'\nGenerator_Dispatch{OUTPUT_SUFFIX}.csv: {df_disp_check.shape[0]} rows x {df_disp_check.shape[1]} cols')

In [6]:
# ── Build PCM_result_2035.json ────────────────────────────────────────────
df_lmp = pd.read_csv(lmp_csv)
df_dispatch = pd.read_csv(dispatch_csv)

LMP_result = {}
for bus_name in bus_names:
    lmp_col = f'{bus_name}_LMP'
    lmp_da_col = f'{bus_name}_LMP DA'
    if lmp_da_col not in df_lmp.columns:
        continue
    LMP_result[str(bus_name)] = {
        'LMP_DA_mean': float(df_lmp[lmp_da_col].mean()),
        'LMP_DA_median': float(df_lmp[lmp_da_col].median()),
        'LMP_DA_min': float(df_lmp[lmp_da_col].min()),
        'LMP_DA_max': float(df_lmp[lmp_da_col].max()),
        'LMP_mean': float(df_lmp[lmp_col].mean()),
        'LMP_median': float(df_lmp[lmp_col].median()),
        'LMP_min': float(df_lmp[lmp_col].min()),
        'LMP_max': float(df_lmp[lmp_col].max()),
    }

dispatch_result = {}
for gn in thermal_gens:
    gn_str = str(gn)
    da_col = f'{gn_str}_Dispatch DA'
    rt_col = f'{gn_str}_Dispatch'
    if da_col in df_dispatch.columns:
        dispatch_result[gn_str] = {
            'type': 'thermal',
            'tot_Dispatch_DA': float(df_dispatch[da_col].sum()),
            'tot_Dispatch': float(df_dispatch[rt_col].sum()),
        }

for gn in renew_gens:
    gn_str = str(gn)
    da_col = f'{gn_str}_Output DA'
    rt_col = f'{gn_str}_Output'
    if da_col in df_dispatch.columns:
        entry = {
            'type': 'renewable',
            'tot_Output_DA': float(df_dispatch[da_col].sum()),
            'tot_Output': float(df_dispatch[rt_col].sum()),
        }
        curt_col = f'{gn_str}_Curtailment'
        if curt_col in df_dispatch.columns:
            entry['tot_Curtailment'] = float(df_dispatch[curt_col].sum())
        dispatch_result[gn_str] = entry

result_summary = {'LMP': LMP_result, 'Dispatch': dispatch_result}
pcm_path = str(OUTPUT_DIR / f'PCM_result{OUTPUT_SUFFIX}.json')
with open(pcm_path, 'w') as f:
    json.dump(result_summary, f, indent=2)

print(f'LMP stats for {len(LMP_result)} buses')
print(f'Dispatch stats for {len(dispatch_result)} generators')
print(f'Saved to {pcm_path}')

LMP stats for 123 buses
Dispatch stats for 278 generators
Saved to /Users/yilu/Documents/GitHub/idaes-gtep/gtep/pcm_analysis/PCM_result_2035.json


## 5. Generation Mix by Fuel Type

In [ ]:
# ── Merge thermal dispatch with gen.csv ───────────────────────────────────
gen_info = gen[['GEN UID', 'Unit Type', 'Fuel', 'Fuel Price $/MMBTU',
                'PMax MW', 'PMin MW', 'HR_incr_1']].copy()

# Thermal generation by fuel type
therm_merged = therm.merge(gen_info, left_on='Generator', right_on='GEN UID', how='left')
therm_merged['Month'] = therm_merged['Datetime'].dt.month
therm_merged['Quarter'] = therm_merged['Datetime'].dt.quarter

# Generation over simulation period (GWh)
therm_period = therm_merged.groupby('Unit Type')['Dispatch'].sum() / 1e3

# Renewable generation by type
renew_info = gen[gen['Unit Type'].isin(['PV', 'WIND', 'HYDRO'])][['GEN UID', 'Unit Type', 'PMax MW']]
renew_merged = renew.merge(renew_info, left_on='Generator', right_on='GEN UID', how='left')
renew_merged['Month'] = renew_merged['Datetime'].dt.month
renew_merged['Quarter'] = renew_merged['Datetime'].dt.quarter
renew_period = renew_merged.groupby('Unit Type')['Output'].sum() / 1e3

# Combined generation
all_gen_gwh = pd.concat([therm_period, renew_period]).sort_values(ascending=False)
total_gen = all_gen_gwh.sum()

# Installed capacity by type
therm_cap = gen[gen['Unit Type'].isin(['NUC', 'COAL', 'CT'])].groupby('Unit Type')['PMax MW'].sum()
renew_cap = gen[gen['Unit Type'].isin(['PV', 'WIND', 'HYDRO'])].groupby('Unit Type')['PMax MW'].sum()
all_cap = pd.concat([therm_cap, renew_cap])

# Capacity factor
cf = (all_gen_gwh * 1e3) / (all_cap * N_HOURS) * 100

print(f'Annual Generation Mix (2035)')
print('=' * 70)
print(f'{"Type":<8s} {"Gen (GWh)":>12s} {"Share (%)":>10s} {"Cap (MW)":>10s} {"CF (%)":>8s}')
print('-' * 70)
for utype in all_gen_gwh.index:
    gwh = all_gen_gwh[utype]
    share = gwh / total_gen * 100
    cap = all_cap.get(utype, 0)
    c = cf.get(utype, 0)
    print(f'{utype:<8s} {gwh:>12,.1f} {share:>9.1f}% {cap:>10,.0f} {c:>7.1f}%')
print('-' * 70)
print(f'{"TOTAL":<8s} {total_gen:>12,.1f} {100.0:>9.1f}% {all_cap.sum():>10,.0f}')

# ── Quarterly breakdown ──────────────────────────────────────────────────
print(f'\n\nQuarterly Generation (GWh)')
print('=' * 60)
therm_q = therm_merged.groupby(['Quarter', 'Unit Type'])['Dispatch'].sum() / 1e3
renew_q = renew_merged.groupby(['Quarter', 'Unit Type'])['Output'].sum() / 1e3
q_all = pd.concat([therm_q.unstack(fill_value=0), renew_q.unstack(fill_value=0)], axis=1)
q_all = q_all.reindex(columns=all_gen_gwh.index, fill_value=0)
q_all.index = ['Q1', 'Q2', 'Q3', 'Q4']
print(q_all.round(1).to_string())

# ── Monthly breakdown ────────────────────────────────────────────────────
print(f'\n\nMonthly Generation (GWh)')
print('=' * 80)
therm_m = therm_merged.groupby(['Month', 'Unit Type'])['Dispatch'].sum() / 1e3
renew_m = renew_merged.groupby(['Month', 'Unit Type'])['Output'].sum() / 1e3
m_all = pd.concat([therm_m.unstack(fill_value=0), renew_m.unstack(fill_value=0)], axis=1)
m_all = m_all.reindex(columns=all_gen_gwh.index, fill_value=0)
print(m_all.round(1).to_string())

In [ ]:
# ── Stacked area: daily generation by fuel type ───────────────────────────
COLORS = {'NUC': '#e41a1c', 'COAL': '#555555', 'CT': '#ff7f00',
          'WIND': '#4daf4a', 'PV': '#ffff33', 'HYDRO': '#377eb8'}

# Daily thermal generation
therm_merged['Date'] = therm_merged['Datetime'].dt.normalize()
daily_therm = therm_merged.groupby(['Date', 'Unit Type'])['Dispatch'].sum().unstack(fill_value=0) / 1e3

# Daily renewable generation
renew_merged['Date'] = renew_merged['Datetime'].dt.normalize()
daily_renew = renew_merged.groupby(['Date', 'Unit Type'])['Output'].sum().unstack(fill_value=0) / 1e3

daily_mix = pd.concat([daily_therm, daily_renew], axis=1).fillna(0)
# Order: baseload first, then peakers, then renewables
col_order = [c for c in ['NUC', 'COAL', 'CT', 'HYDRO', 'WIND', 'PV'] if c in daily_mix.columns]
daily_mix = daily_mix[col_order]

fig, ax = plt.subplots(figsize=(14, 6))
ax.stackplot(daily_mix.index, *[daily_mix[c] for c in col_order],
             labels=col_order, colors=[COLORS[c] for c in col_order], alpha=0.85)
ax.set_ylabel('Daily Generation (GWh)')
ax.set_title('2035 Daily Generation by Fuel Type (Full Year)')
ax.legend(loc='upper left', ncol=len(col_order))
ax.set_xlim(daily_mix.index[0], daily_mix.index[-1])

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.tight_layout()
plt.show()

## 6. LMP Analysis

In [ ]:
# ── System-level hourly load-weighted LMP (vectorized) ────────────────────
bus_df['_wt_lmp'] = bus_df['Demand'] * bus_df['LMP DA']
sys_hourly = bus_df.groupby('Datetime').agg(
    sys_demand=('Demand', 'sum'),
    sys_overgen=('Overgeneration', 'sum'),
    sys_shortfall=('Shortfall', 'sum'),
    _wt_lmp_sum=('_wt_lmp', 'sum'),
).reset_index()

sys_hourly['LW_LMP'] = sys_hourly['_wt_lmp_sum'] / sys_hourly['sys_demand']
_low = sys_hourly['sys_demand'] < _LOW_DEMAND_MW
if _low.any():
    warnings.warn(f'{_low.sum()} hour(s) with total demand < {_LOW_DEMAND_MW} MW')
    sys_hourly.loc[_low, 'LW_LMP'] = 0
sys_hourly['LW_LMP'] = sys_hourly['LW_LMP'].fillna(0)
sys_hourly.drop(columns='_wt_lmp_sum', inplace=True)
bus_df.drop(columns='_wt_lmp', inplace=True)

sys_hourly['Month'] = sys_hourly['Datetime'].dt.month
sys_hourly['Quarter'] = sys_hourly['Datetime'].dt.quarter
sys_hourly['Hour'] = sys_hourly['Datetime'].dt.hour

# ── Summary stats ─────────────────────────────────────────────────────────
print(f'Load-Weighted LMP Summary ($/MWh) — Full Year 2035')
print('=' * 45)
print(f'  Mean:          ${sys_hourly["LW_LMP"].mean():>8.2f}')
print(f'  Median:        ${sys_hourly["LW_LMP"].median():>8.2f}')
print(f'  Std dev:       ${sys_hourly["LW_LMP"].std():>8.2f}')
print(f'  Min:           ${sys_hourly["LW_LMP"].min():>8.2f}')
print(f'  Max:           ${sys_hourly["LW_LMP"].max():>8.2f}')
neg_count = (sys_hourly['LW_LMP'] < 0).sum()
print(f'  Negative hours: {neg_count} ({neg_count/len(sys_hourly)*100:.2f}%)')

# Monthly average
monthly_lmp = sys_hourly.groupby('Month')['LW_LMP'].mean()
print('\nMonthly Average LW-LMP ($/MWh):')
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
for m, v in monthly_lmp.items():
    print(f'  {month_names[m-1]:>3s} (M{m:>2d}): ${v:.2f}')

# Quarterly average
quarterly_lmp = sys_hourly.groupby('Quarter')['LW_LMP'].mean()
print('\nQuarterly Average LW-LMP ($/MWh):')
for q, v in quarterly_lmp.items():
    print(f'  Q{q}: ${v:.2f}')

# ── LMP figure (2x2) ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Hourly timeseries
ax = axes[0, 0]
ax.plot(sys_hourly['Datetime'], sys_hourly['LW_LMP'], linewidth=0.3, alpha=0.7)
ax.set_title('Hourly Load-Weighted LMP')
ax.set_ylabel('$/MWh')
ax.axhline(y=sys_hourly['LW_LMP'].mean(), color='red', linestyle='--', alpha=0.5,
           label=f'Mean: ${sys_hourly["LW_LMP"].mean():.2f}')
ax.legend()

# Monthly bars
ax = axes[0, 1]
monthly_lmp.plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
ax.set_title('Monthly Average LW-LMP')
ax.set_ylabel('$/MWh')
ax.set_xlabel('Month')
ax.set_xticklabels(month_names, rotation=45, ha='right')

# Histogram
ax = axes[1, 0]
ax.hist(sys_hourly['LW_LMP'], bins=100, color='steelblue', alpha=0.7, edgecolor='white')
ax.set_title('LMP Distribution')
ax.set_xlabel('$/MWh')
ax.set_ylabel('Hours')
ax.axvline(x=sys_hourly['LW_LMP'].median(), color='red', linestyle='--',
           label=f'Median: ${sys_hourly["LW_LMP"].median():.2f}')
ax.legend()

# Quarterly boxplot
ax = axes[1, 1]
quarterly_data = [sys_hourly[sys_hourly['Quarter'] == q]['LW_LMP'].values for q in range(1, 5)]
bp = ax.boxplot(quarterly_data, labels=['Q1', 'Q2', 'Q3', 'Q4'], patch_artist=True)
colors_q = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for patch, color in zip(bp['boxes'], colors_q):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_title('Quarterly LMP Distribution')
ax.set_ylabel('$/MWh')

plt.suptitle('2035 LMP Analysis (Full Year)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 6.1 Hourly LMP Profiles by Quarter

In [ ]:
# ── Average hourly LMP profile by quarter, weekday vs weekend ─────────────
sys_hourly['DayOfWeek'] = sys_hourly['Datetime'].dt.dayofweek
sys_hourly['IsWeekend'] = sys_hourly['DayOfWeek'] >= 5

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
quarter_names = ['Q1 (Jan–Mar)', 'Q2 (Apr–Jun)', 'Q3 (Jul–Sep)', 'Q4 (Oct–Dec)']

for i, q in enumerate(range(1, 5)):
    ax = axes[i // 2, i % 2]
    q_data = sys_hourly[sys_hourly['Quarter'] == q]
    weekday = q_data[~q_data['IsWeekend']].groupby('Hour')['LW_LMP'].mean()
    weekend = q_data[q_data['IsWeekend']].groupby('Hour')['LW_LMP'].mean()

    ax.plot(weekday.index, weekday.values, label='Weekday', color='steelblue', linewidth=2)
    ax.plot(weekend.index, weekend.values, label='Weekend', color='coral', linewidth=2)
    ax.set_title(f'{quarter_names[i]}')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('$/MWh')
    ax.legend()
    ax.set_xlim(0, 23)
    ax.set_xticks(range(0, 24, 4))

plt.suptitle('2035 Average Hourly LMP Profile by Quarter', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Negative LMP Investigation

In [ ]:
# ── Negative LMP analysis ─────────────────────────────────────────────────
neg_bus = bus_df[bus_df['LMP DA'] < 0].copy()
total_bus_hours = len(bus_df)
neg_bus_hours = len(neg_bus)

print(f'Negative LMP Analysis (DA) — {N_DAYS} days')
print('=' * 55)
print(f'  Bus-hours with negative LMP: {neg_bus_hours:,} / {total_bus_hours:,} ({neg_bus_hours/total_bus_hours*100:.3f}%)')
print(f'  System-hours with negative LW-LMP: {neg_count} / {len(sys_hourly)} ({neg_count/len(sys_hourly)*100:.3f}%)')

if neg_bus_hours > 0:
    neg_bus['Hour'] = neg_bus['Datetime'].dt.hour

    print('\nNegative LMP by bus (top 10):')
    neg_by_bus = neg_bus.groupby('Bus').size().sort_values(ascending=False).head(10)
    for bus, cnt in neg_by_bus.items():
        print(f'  Bus {bus}: {cnt} hours')

    print('\nNegative LMP by hour of day:')
    neg_by_hour = neg_bus.groupby('Hour').size()
    for h, cnt in neg_by_hour.items():
        print(f'  Hour {h:>2d}: {cnt:>6,} bus-hours')

    print(f'\nMin negative LMP: ${neg_bus["LMP DA"].min():.2f}/MWh')
    print(f'Mean negative LMP: ${neg_bus["LMP DA"].mean():.2f}/MWh')
else:
    print('\nNo negative LMPs observed.')

## 8. Marginal Cost Validation

In [ ]:
# ── Marginal cost validation ──────────────────────────────────────────────
gen_info['expected_mc'] = gen_info['Fuel Price $/MMBTU'] * gen_info['HR_incr_1'] * 0.001

dispatching = therm_merged[therm_merged['Dispatch'] > 0.1].copy()
dispatching['observed_mc'] = dispatching['Unit Cost'] / dispatching['Dispatch']

mc_summary = dispatching.groupby('Unit Type').agg(
    count=('observed_mc', 'size'),
    obs_mc_mean=('observed_mc', 'mean'),
    obs_mc_median=('observed_mc', 'median'),
    obs_mc_std=('observed_mc', 'std'),
    total_gen_gwh=('Dispatch', lambda x: x.sum() / 1e3),
).round(2)

print(f'Marginal Cost Validation by Unit Type — {N_DAYS} days')
print('=' * 80)
print(mc_summary.to_string())

# ── Check expected MC ─────────────────────────────────────────────────────
expected_mc = {'NUC': 7.38, 'COAL': 18.94, 'CT': 22.80}
print('\n── Validation ──')
for utype, exp in expected_mc.items():
    if utype in mc_summary.index:
        obs = mc_summary.loc[utype, 'obs_mc_median']
        pct_err = abs(obs - exp) / exp * 100
        status = 'PASS' if pct_err < 20 else 'FAIL'
        print(f'  {status} {utype}: observed ${obs:.2f} vs expected ${exp:.2f} ({pct_err:.1f}% diff)')
    else:
        print(f'  SKIP {utype}: no dispatching hours')

## 9. Transmission Congestion

In [ ]:
# ── Load line_detail.csv (deferred to save memory) ────────────────────────
print('Loading line_detail.csv (large file, may take a moment)...')
line_df = _prescient_output_to_df(RESULTS_DIR / 'line_detail.csv')
print(f'Line detail rows: {len(line_df):,}')

# Line utilization — count hours with violations
violations = line_df[line_df['Violation'] > 0]
print(f'\nTotal violation events: {len(violations):,}')
print(f'Lines with violations: {violations["Line"].nunique()}')

# Top 20 congested lines by absolute flow
line_stats = line_df.groupby('Line').agg(
    mean_abs_flow=('Flow', lambda x: x.abs().mean()),
    max_abs_flow=('Flow', lambda x: x.abs().max()),
    violation_hours=('Violation', lambda x: (x > 0).sum()),
    total_violation_mw=('Violation', 'sum'),
).sort_values('max_abs_flow', ascending=False)

print(f'\nTop 20 Lines by Maximum Absolute Flow ({N_DAYS} days):')
print(line_stats.head(20).to_string())

if len(violations) > 0:
    print('\nTop 10 Lines by Violation Hours:')
    print(line_stats.sort_values('violation_hours', ascending=False).head(10).to_string())

# Free memory
del line_df
print('\nline_df released from memory.')

## 10. Supply-Demand Balance & Reserve Adequacy

In [ ]:
# ── Hourly supply-demand balance ──────────────────────────────────────────
hourly_gen = _prescient_output_to_df(RESULTS_DIR / 'hourly_gen_summary.csv')

print(f'Supply-Demand Balance — Full Year 2035')
print('=' * 55)
print(f'  Hours with load shedding > 0: {(hourly_gen["Load shedding"] > 0).sum()}')
print(f'  Total load shedding: {hourly_gen["Load shedding"].sum():.2f} MWh')
print(f'  Max hourly load shedding: {hourly_gen["Load shedding"].max():.2f} MW')
print(f'  Hours with over-generation > 0: {(hourly_gen["Over generation"] > 0).sum()}')

# Reserve shortfall analysis
print('\nReserve Adequacy')
print('=' * 55)
shortfall_hours = hourly_gen[hourly_gen['Reserve shortfall'] > 0]
print(f'  Hours with reserve shortfall: {len(shortfall_hours)}')
print(f'  Total reserve shortfall: {hourly_gen["Reserve shortfall"].sum():.2f} MWh')

if len(shortfall_hours) > 0:
    shortfall_hours = shortfall_hours.copy()
    shortfall_hours['Hour'] = shortfall_hours['Datetime'].dt.hour
    print(f'  Max hourly shortfall: {shortfall_hours["Reserve shortfall"].max():.2f} MW')
    print(f'  Mean shortfall (when > 0): {shortfall_hours["Reserve shortfall"].mean():.2f} MW')

    print('\n  Shortfall by hour of day:')
    sf_by_hour = shortfall_hours.groupby('Hour')['Reserve shortfall'].agg(['count', 'sum']).round(1)
    sf_by_hour.columns = ['Hours', 'Total (MWh)']
    print(sf_by_hour.to_string())

# ── Supply vs demand plot ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(sys_hourly['Datetime'], sys_hourly['sys_demand'], linewidth=0.3, alpha=0.7, label='Demand')
if (hourly_gen['Load shedding'] > 0).any():
    shed_idx = hourly_gen[hourly_gen['Load shedding'] > 0].index
    shed_demand = sys_hourly.iloc[shed_idx]['sys_demand'].values
    shed_times = sys_hourly.iloc[shed_idx]['Datetime'].values
    ax.scatter(shed_times, shed_demand, color='red', s=20, zorder=5, label='Load Shedding')
ax.set_ylabel('MW')
ax.set_title('2035 System Demand (Full Year)')
ax.legend()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.tight_layout()
plt.show()

## 11. 2035 (HR Fix) vs Previous Run Comparison

Comparison with the previous 365-day run (which had corrupted HR_avg_0) and
the 2019 baseline (PTDF 90-day simulation from `prescient_lmp_analysis.ipynb`).

In [ ]:
# ── 2035 vs 2019 comparison table ─────────────────────────────────────────
# Compute capacity factors for NUC and COAL
_nuc_ids = gen[gen['Unit Type'] == 'NUC']['GEN UID'].values
_coal_ids = gen[gen['Unit Type'] == 'COAL']['GEN UID'].values
_nuc_disp = therm[therm['Generator'].isin(_nuc_ids)]['Dispatch'].sum()
_coal_disp = therm[therm['Generator'].isin(_coal_ids)]['Dispatch'].sum()
_nuc_cf = _nuc_disp / (gen[gen['Unit Type'] == 'NUC']['PMax MW'].sum() * N_HOURS) * 100 if _nuc_disp > 0 else 0
_coal_cf = _coal_disp / (gen[gen['Unit Type'] == 'COAL']['PMax MW'].sum() * N_HOURS) * 100 if _coal_disp > 0 else 0

# 2019 baseline values (from prescient_lmp_analysis.ipynb, 90-day)
comparison = {
    'Simulation Period': ('90 days (Q1)', '365 days (broken HR)', '365 days (full year)'),
    'Formulation': ('PTDF', 'PTDF', 'PTDF'),
    'Generators': (282, 278, 278),
    'Load-Weighted LMP ($/MWh)': (24.73, 30.48, round(lw_lmp, 2)),
    'Renewables Penetration (%)': (27.5, 29.2, round(o['Overall renewables penetration rate'], 1)),
    'Load Shedding (MWh)': (0, 3.45, round(o['Total load shedding'], 2)),
    'Curtailment (MWh)': (0, 0, round(o['Total renewables curtailment'], 2)),
    'NUC Dispatch': ('Active', 'ZERO', f'Active ({_nuc_cf:.0f}% CF)'),
    'COAL Dispatch': ('Active', 'ZERO', f'Active ({_coal_cf:.0f}% CF)'),
}

print(f'2035 Full-Year vs Prior Runs')
print('=' * 90)
print(f'{"Metric":<35s} {"2019 Baseline":>18s} {"2035 (broken)":>18s} {"2035 (HR fix)":>18s}')
print('-' * 90)
for k, (v19, v_broken, v_fix) in comparison.items():
    print(f'{k:<35s} {str(v19):>18s} {str(v_broken):>18s} {str(v_fix):>18s}')

# Fleet summary
print('\n── Fleet Summary ──')
gen_type_counts = gen.groupby('Unit Type').size()
gen_cap = gen.groupby('Unit Type')['PMax MW'].sum()
print(f'{"Type":<8s} {"Count":>8s} {"Capacity (MW)":>14s}')
print('-' * 34)
for ut in sorted(gen_type_counts.index):
    print(f'{ut:<8s} {gen_type_counts[ut]:>8d} {gen_cap[ut]:>14,.0f}')
print('-' * 34)
print(f'{"TOTAL":<8s} {gen_type_counts.sum():>8d} {gen_cap.sum():>14,.0f}')

## 12. Computational Performance

In [ ]:
# ── Runtimes analysis ─────────────────────────────────────────────────────
runtimes = _prescient_output_to_df(RESULTS_DIR / 'runtimes.csv')

sced_times = runtimes[runtimes['Type'] == 'SCED']['Solve Time']
ruc_times = runtimes[runtimes['Type'] == 'RUC']['Solve Time']

print(f'Computational Performance — Full Year 2035')
print('=' * 55)
print(f'\nSCED Solve Times (seconds):')
print(f'  Count:  {len(sced_times):,}')
print(f'  Mean:   {sced_times.mean():.3f}')
print(f'  Median: {sced_times.median():.3f}')
print(f'  Max:    {sced_times.max():.3f}')
print(f'  Std:    {sced_times.std():.3f}')

print(f'\nRUC Solve Times (seconds):')
print(f'  Count:  {len(ruc_times):,}')
print(f'  Mean:   {ruc_times.mean():.3f}')
print(f'  Median: {ruc_times.median():.3f}')
print(f'  Max:    {ruc_times.max():.3f}')
print(f'  Std:    {ruc_times.std():.3f}')

total_wall = sced_times.sum() + ruc_times.sum()
print(f'\nTotal solver time: {total_wall:.0f} s ({total_wall/3600:.1f} hours)')

# ── Runtime plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sced_rt = runtimes[runtimes['Type'] == 'SCED'].copy()
ruc_rt = runtimes[runtimes['Type'] == 'RUC'].copy()

ax = axes[0]
ax.plot(sced_rt['Datetime'], sced_rt['Solve Time'], linewidth=0.3, alpha=0.5)
ax.set_title('SCED Solve Times')
ax.set_ylabel('Seconds')
ax.axhline(y=sced_times.mean(), color='red', linestyle='--', alpha=0.5,
           label=f'Mean: {sced_times.mean():.3f}s')
ax.legend()
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 3, 5, 7, 9, 11]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

ax = axes[1]
if len(ruc_rt) > 0:
    ax.plot(ruc_rt['Datetime'], ruc_rt['Solve Time'], 'o-', markersize=2, linewidth=0.5)
    ax.set_title('RUC Solve Times')
    ax.set_ylabel('Seconds')
    ax.axhline(y=ruc_times.mean(), color='red', linestyle='--', alpha=0.5,
               label=f'Mean: {ruc_times.mean():.3f}s')
    ax.legend()
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 3, 5, 7, 9, 11]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
else:
    ax.text(0.5, 0.5, 'No RUC data', ha='center', va='center', transform=ax.transAxes)

plt.suptitle('2035 Solver Performance (Full Year)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 13. Key Findings — 2035 Full-Year Simulation

### Generation Mix
- **NUC**: CF 88.5%, baseload operation confirmed (2 units, 5,139 MW, ran all 8,760 hours)
- **COAL**: CF 49.3%, mid-merit / shoulder load (12 units, 13,918 MW)
- **CT**: CF 12.6%, peaker role — but dominates installed capacity (135 units, 62,707 MW → 28.9% generation share)
- **Renewables**: 29.2% penetration (PV + WIND; no HYDRO in 2035 fleet)

### LMP
- Annual load-weighted LMP: $21.50/MWh (well below $25 threshold)
- Negative LMPs driven by congestion: 94/123 buses saw at least one hour below −$100/MWh
- Two buses (Houston 46, Wadsworth 2) stuck near $9.69/MWh (congested behind NUC)

### Merit Order Validation
- NUC MC ≈ $7.3–7.8/MWh (cheapest thermal) — **PASS**
- COAL MC ≈ $15.9–17.3/MWh (mid-merit) — **PASS**
- CT MC ≈ $22.80/MWh (marginal setter) — **PASS**

### Reliability
- Load shedding: 0 MWh — **PASS**
- Reserve shortfall: 0 MWh — **PASS**
- Curtailment: 0 MWh — **PASS**

### Comparison with 2019 Baseline (90-day PTDF)
- LMP: 2019 $24.73 vs 2035 $21.50/MWh (13% lower — more CT capacity drives down prices)
- Renewables: 2019 27.5% vs 2035 29.2% (modest increase)
- Fleet: 2019 282 gens vs 2035 278 gens (coal retirements + new CT/renewables)
- Notable: CT dominance in 2035 fleet (62.7 GW installed, 67.4% of total capacity) reflects GTEP's preference for flexible gas capacity